# 01 — Exploration des données (EDA)

**Projet** : Prévision de consommation électrique multi-horizons avec scikit-learn
**Cas métier** : Prédire la consommation électrique quotidienne régionale (en MW) pour les horizons J+1, J+2, J+3 et J+7, à partir de l'histoire de consommation disponible **au matin de l'origine** et de la prévision météo connue à ce même instant, puis publier une prévision assortie d'un intervalle et d'un indicateur de confiance.
**Jeu de données** : `regional_electricity_load` — Consommation électrique régionale et prévision multi-horizons

> Jeu d'apprentissage supervisé construit par **expansion temporelle** d'une série quotidienne de consommation régionale (~3,4 ans) : chaque ligne correspond à un couple (origine, horizon) et porte la consommation réellement observée `horizon_days` jours après l'origine, ainsi que les seules informations disponibles au matin de cette origine. La série sous-jacente est générée par un modèle explicite et réaliste : niveau de base, tendance lente, saisonnalité annuelle (pointe hivernale), saisonnalité hebdomadaire (creux du week-end), thermo-sensibilité asymétrique (3,5 % par degré sous 15 °C, trois fois moins au-dessus de 24 °C car le parc français est peu climatisé), jours fériés et vacances scolaires, épisodes extrêmes (vague de froid, canicule, arrêt industriel) et bruit autocorrélé AR(1). La prévision de température fournie au modèle est la vérité **entachée d'une erreur de prévision croissante avec l'horizon**, comme dans la réalité : c'est ce qui rend la dégradation du J+1 au J+7 incompressible.

## Objectifs pédagogiques

1. Charger un jeu de données tabulaire et en établir le profil (types, manquants, doublons).
1. Lire une distribution : détecter déséquilibre, outliers et colinéarité **avant** de modéliser.
1. Relier chaque observation statistique à une conséquence métier ou de modélisation.
1. Produire les figures qui serviront de référence dans les notebooks suivants.

**Objectifs transverses du dépôt**

- Construire un jeu supervisé par expansion temporelle (origine x horizon) et formaliser le contrat d'antériorité de chaque feature : connue à l'origine, connue par avance, ou interdite.
- Comprendre pourquoi un split chronologique s'impose et ce que coûte concrètement une validation croisée aléatoire sur une série temporelle.
- Comparer un modèle appris à trois références triviales (persistance, naif saisonnier, moyenne glissante) et quantifier la valeur ajoutée réelle plutôt que le R².

## 0. Environnement

Toute la configuration vient de **Hydra** (`conf/`) : aucune valeur métier n'est codée en dur
dans ce notebook. Si `data/raw` est vide, le générateur synthétique du projet prend le relais
(voir `make data`).

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous les
# lignes INFO de production. Les avertissements réels restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (4800 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 4800

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 4.0)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

**Pourquoi ce bloc d'initialisation**

- `CONFIG` est l'objet **Pydantic** validé : une clé incohérente échoue ici, pas en production.
- Les notebooks travaillent sur un échantillon réduit pour rester rapides ; `make train` utilise `data.n_samples` complet.
- `NB_PATHS` isole les écritures du notebook dans `outputs/notebooks`.

## 1. Chargement et premier contact

On ne regarde jamais un dataset sans vérifier trois choses : sa **forme** (lignes x colonnes),
ses **types** (un numérique lu comme texte casse tout) et ses **premières lignes** (les valeurs
ont-elles du sens métier ?).

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

**Ce qu'il faut retenir**

- Le contrat `regional_electricity_load` est documenté dans `data/README.md` : chaque colonne y a une signification métier.
- Les identifiants et horodatages ne sont **pas** des features : ils servent à tracer et à splitter.

In [ ]:
from src.data.schemas import describe_schema, validation_report

# Types déclarés (contrat Pandera) vs types réellement lus : toute divergence est un signal.
contract = describe_schema("raw")
observed = pd.DataFrame({"dtype_lu": {str(k): str(v) for k, v in raw.dtypes.items()}})
contract.join(observed)[["dtype", "dtype_lu", "nullable", "unique", "checks"]]

**Ce qu'il faut retenir**

- La colonne `dtype` vient du **contrat**, `dtype_lu` de la source : elles doivent correspondre.
- Les `checks` (bornes, valeurs autorisées) sont la mémoire des règles métier — ils seront testés au notebook 02.

In [ ]:
report = validation_report(raw)
summary = pd.DataFrame(
    {
        "indicateur": [
            "lignes",
            "colonnes",
            "cellules manquantes",
            "taux de manquants",
            "mémoire (Ko)",
        ],
        "valeur": [
            report["n_rows"],
            report["n_columns"],
            report["missing_cells"],
            f"{report['missing_rate']:.2%}",
            round(report["memory_kb"], 1),
        ],
    }
)
summary

**Ce qu'il faut retenir**

- Un taux de manquants global faible peut cacher une colonne très incomplète : regarder **par colonne**.
- La mémoire indique si le dataset tient en RAM (sinon : pyarrow, chunking ou échantillonnage).

## 2. Valeurs manquantes

Où, combien, et surtout : **manquant au hasard ou pas** ? Un manquant informatif (ex. score de satisfaction non renseigné par les clients mécontents) est un signal, pas seulement un problème technique.

In [ ]:
missing = raw.isna().sum()
missing_frame = (
    pd.DataFrame({"manquants": missing, "taux": (missing / len(raw)).round(4)})
    .loc[lambda frame: frame["manquants"] > 0]
    .sort_values("manquants", ascending=False)
)
missing_frame

In [ ]:
if missing_frame.empty:
    print("Aucune valeur manquante dans cet échantillon.")
else:
    fig, axis = plt.subplots(figsize=(7.5, 0.55 * len(missing_frame) + 1.6))
    axis.barh(missing_frame.index[::-1], missing_frame["taux"][::-1] * 100, color="#d1495b")
    axis.set_xlabel("Cellules manquantes (%)")
    axis.set_title("Valeurs manquantes par colonne")
    fig.tight_layout()
    plt.show()

**Ce qu'il faut retenir**

- L'imputation doit être **apprise sur le train** (moyenne/médiane/constante) puis appliquée aux autres splits.
- Ajouter un indicateur binaire « valeur manquante » est souvent rentable quand le manquant est informatif.
- Notes du générateur : Construction par expansion temporelle : 1 200 origines x 4 horizons = 4 800 lignes. Les quatre lignes d'une même origine partagent leur histoire et diffèrent par le jour cible.; Contract d'antériorité : `load_lag_*`, `load_rolling_*` et `temperature_lag/rolling` sont calculés **jusqu'à l'origine incluse** ; `target_*` (calendrier) et `temperature_forecast_c` sont connus par avance ; `load_mw`, `target_date`, `event_type` et `is_extreme_event` sont des métadonnées de vérité terrain..

## 3. Distributions numériques

In [ ]:
numeric_columns = [column for column in raw.columns if pd.api.types.is_numeric_dtype(raw[column])]
numeric_columns = [column for column in numeric_columns if column != CONFIG.data.target]

n_plots = len(numeric_columns)
n_cols = 3
n_rows = int(np.ceil(n_plots / n_cols)) if n_plots else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 2.7 * n_rows))
for axis, column in zip(np.atleast_1d(axes).ravel(), numeric_columns, strict=False):
    raw[column].hist(bins=30, ax=axis, color="#005f73", edgecolor="white")
    axis.set_title(column, fontsize=9)
    axis.tick_params(labelsize=7)
for axis in np.atleast_1d(axes).ravel()[len(numeric_columns) :]:
    axis.axis("off")
fig.suptitle("Distributions des variables numériques", y=1.005)
fig.tight_layout()
plt.show()

In [ ]:
raw[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2)

**Ce qu'il faut retenir**

- Une distribution très asymétrique (max ≫ p99) justifie un **winsorising** ou un `log1p` plutôt qu'une suppression d'outliers.
- Des échelles hétérogènes (euros, Go, unités) imposent un **scaling** pour les modèles sensibles à la distance (SVM, k-NN, réseaux).
- Comparer `mean` et `50%` : un écart important signale une queue lourde.

## 4. Variables catégorielles

In [ ]:
categorical_columns = [
    column
    for column in raw.columns
    if not pd.api.types.is_numeric_dtype(raw[column])
    and column not in [*CONFIG.data.drop_columns, str(CONFIG.data.target)]
]

for column in categorical_columns:
    counts = raw[column].astype(str).value_counts()
    print(f"--- {column} ({len(counts)} modalités) ---")
    print((counts / len(raw)).map("{:.1%}".format).to_string())

**Ce qu'il faut retenir**

- Une modalité ultra-rare (< 1 %) doit être regroupée dans un bucket `rare` : sinon l'encodage one-hot crée des colonnes quasi vides et instables.
- Une cardinalité élevée (identifiants, codes postaux) appelle un **target encoding** régularisé plutôt qu'un one-hot.

## 5. Structure temporelle : ce qui se prévoit, et avec quoi

Une prévision de consommation n'est pas un problème tabulaire ordinaire : chaque ligne est un
**couple (origine, horizon)** et la cible est la consommation d'un jour **futur**. Trois
conséquences structurent toute l'analyse qui suit :

1. la série porte **plusieurs saisonnalités superposées** (annuelle, hebdomadaire, et une composante
   météo qui n'est pas saisonnière) ;
2. la relation à la température est **asymétrique** — le chauffage pèse beaucoup plus que la
   climatisation dans le parc français ;
3. le jeu contient des **régimes exceptionnels** (vague de froid, canicule, arrêt industriel) qui
   sont rares en jours mais dominants en coût d'erreur.

Et une règle absolue, vérifiée chiffre à l'appui en section 5.4 : **rien dans les features ne doit
dépendre de la vérité du jour cible**.

### 5.1 La série quotidienne et ses saisonnalités

In [ ]:
# --- Contrat de prévision : tout est lu dans la configuration, rien n'est codé en dur -----------
FORECAST_CONF = dict(CONFIG.model_dump().get("load_forecasting") or {})
HORIZON_COLUMN = str(FORECAST_CONF.get("horizon_column") or "horizon_days")
HORIZONS = tuple(int(value) for value in (FORECAST_CONF.get("horizons") or ()))
LONG_HORIZON = int(FORECAST_CONF.get("long_horizon") or (max(HORIZONS) if HORIZONS else 7))
INTERVAL_LEVEL = float(FORECAST_CONF.get("interval_level") or 0.90)
INTERVAL_METHOD = str(FORECAST_CONF.get("interval_method") or "normalized_conformal")
INTERVAL_SCALE = str(FORECAST_CONF.get("interval_scale_column") or "load_last_observed")
BACKTEST_FOLDS = int(FORECAST_CONF.get("backtest_folds") or 5)

TARGET = str(CONFIG.data.target)
TIME_COLUMN = str(CONFIG.data.time_column or "origin_date")
TARGET_DATE = "target_date" if "target_date" in raw.columns else TIME_COLUMN
EVENT_COLUMN = "event_type" if "event_type" in raw.columns else None
NAIVE_COLUMN = "load_seasonal_naive" if "load_seasonal_naive" in raw.columns else None
PERSIST_COLUMN = "load_last_observed" if "load_last_observed" in raw.columns else None
TEMP_FORECAST = "temperature_forecast_c" if "temperature_forecast_c" in raw.columns else None


def mape(truth: Any, predicted: Any) -> float:
    """Mean absolute percentage error, in percent, ignoring zero denominators.

    Args:
        truth: Observed values.
        predicted: Forecast values.

    Returns:
        The MAPE in percent (``nan`` when nothing is measurable).
    """
    observed = np.asarray(truth, dtype="float64")
    forecast = np.asarray(predicted, dtype="float64")
    usable = np.isfinite(observed) & np.isfinite(forecast) & (np.abs(observed) > 1e-8)
    if not usable.any():
        return float("nan")
    return float(np.mean(np.abs((observed[usable] - forecast[usable]) / observed[usable])) * 100.0)


def mase(truth: Any, predicted: Any, reference: Any) -> float:
    """Mean absolute scaled error: model error over the naive reference error.

    Args:
        truth: Observed values.
        predicted: Forecast values.
        reference: Naive reference forecast on the same rows.

    Returns:
        The MASE (below 1 means better than the reference).
    """
    observed = np.asarray(truth, dtype="float64")
    forecast = np.asarray(predicted, dtype="float64")
    naive = np.asarray(reference, dtype="float64")
    usable = np.isfinite(observed) & np.isfinite(forecast) & np.isfinite(naive)
    scale = float(np.mean(np.abs(observed[usable] - naive[usable])))
    if not usable.any() or scale < 1e-9:
        return float("nan")
    return float(np.mean(np.abs(observed[usable] - forecast[usable])) / scale)


def daily_series(frame: pd.DataFrame) -> pd.DataFrame:
    """Collapse the (origin, horizon) panel into one row per target day.

    Le panel contient plusieurs lignes par jour cible (une par horizon) qui portent **la même**
    consommation : la série quotidienne se reconstruit en dédupliquant sur la date cible.

    Args:
        frame: Raw panel.

    Returns:
        One row per target day, sorted chronologically.
    """
    unique = frame.drop_duplicates(subset=[TARGET_DATE]).copy()
    unique[TARGET_DATE] = pd.to_datetime(unique[TARGET_DATE])
    return unique.sort_values(TARGET_DATE).reset_index(drop=True)


print(
    f"contrat de prévision : horizons={HORIZONS} colonne='{HORIZON_COLUMN}' "
    f"intervalle={INTERVAL_METHOD} (niveau {INTERVAL_LEVEL:.0%}, échelle '{INTERVAL_SCALE}')"
)
print(
    f"cible='{TARGET}' origine='{TIME_COLUMN}' cible_date='{TARGET_DATE}' "
    f"régimes='{EVENT_COLUMN}' naif='{NAIVE_COLUMN}'"
)

# La série quotidienne : trois saisonnalités superposées et une rupture de régime.
daily = daily_series(raw)
daily["doy"] = daily[TARGET_DATE].dt.dayofyear
daily["weekday"] = daily[TARGET_DATE].dt.dayofweek

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
axes[0].plot(daily[TARGET_DATE], daily[TARGET], lw=0.7, color="#1f4e79")
axes[0].set_title(f"Série quotidienne ({len(daily)} jours) — cible `{TARGET}`")
axes[0].set_ylabel("MW")
axes[0].grid(alpha=0.3)

weekly = daily.groupby("weekday")[TARGET].mean()
axes[1].bar(
    ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"],
    weekly.reindex(range(7)).to_numpy(),
    color="#2e75b6",
)
axes[1].set_title("Profil hebdomadaire (moyenne par jour de la semaine)")
axes[1].set_ylabel("MW")
axes[1].grid(alpha=0.3, axis="y")

monthly = daily.groupby(daily[TARGET_DATE].dt.month)[TARGET].mean()
axes[2].plot(monthly.index, monthly.to_numpy(), marker="o", color="#c00000")
axes[2].set_title("Profil annuel (moyenne par mois)")
axes[2].set_ylabel("MW")
axes[2].set_xlabel("mois")
axes[2].grid(alpha=0.3)
fig.tight_layout()
plt.show()

print(f"période couverte : {daily[TARGET_DATE].min().date()} -> {daily[TARGET_DATE].max().date()}")
print(f"nombre d'années civiles : {daily[TARGET_DATE].dt.year.nunique()}")
print(
    f"niveau : moyenne={daily[TARGET].mean():.0f} MW, médiane={daily[TARGET].median():.0f} MW, "
    f"min={daily[TARGET].min():.0f}, max={daily[TARGET].max():.0f}"
)
print(
    f" amplitude semaine : {weekly.max() - weekly.min():.0f} MW "
    f"({100 * (weekly.max() - weekly.min()) / weekly.mean():.1f} % du niveau moyen)"
)
print(
    f" amplitude année   : {monthly.max() - monthly.min():.0f} MW "
    f"({100 * (monthly.max() - monthly.min()) / monthly.mean():.1f} % du niveau moyen)"
)

**Ce qu'il faut retenir**

- La série est **non stationnaire à trois échelles** : une tendance lente (effacement, efficacité énergétique), un cycle annuel marqué (chauffage) et un cycle hebdomadaire (tertiaire fermé le week-end). Un modèle qui ne verrait que le niveau moyen perdrait l'essentiel.
- L'amplitude annuelle est très supérieure à l'amplitude hebdomadaire : c'est la thermo-sensibilité qui pilote le niveau, le calendrier pilote la forme. Cela fixe l'ordre des priorités en feature engineering.
- Le panel contient plusieurs lignes par jour cible (une par horizon) : la reconstruction de la série quotidienne passe par une déduplication sur la date cible, ce que fait `daily_series()`.

### 5.2 Thermo-sensibilité : une courbe, pas une droite

La température vraie du jour cible **n'est pas dans le jeu** — ce serait une fuite. On travaille
donc avec la prévision de température, entachée de son erreur réelle, qui est exactement ce dont
dispose le modèle le matin de la publication.

In [ ]:
# Thermo-sensibilité : la relation consommation / température, et son asymétrie.
# La température **vraie** du jour cible n'est volontairement pas dans le jeu (contrat
# d'antériorité) : on travaille avec la prévision de température, qui est ce dont dispose le modèle.
if TEMP_FORECAST is None:
    print("aucune colonne de prévision de température dans ce jeu : section sans objet")
else:
    panel = raw.drop_duplicates(subset=[TARGET_DATE]).copy()
    bins = np.arange(-15, 40, 2.5)
    panel["temp_bin"] = pd.cut(panel[TEMP_FORECAST], bins=bins)
    profile = panel.groupby("temp_bin", observed=True).agg(
        jours=(TARGET, "size"),
        consommation=(TARGET, "mean"),
        ecart_type=(TARGET, "std"),
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].scatter(panel[TEMP_FORECAST], panel[TARGET], s=4, alpha=0.25, color="#2e75b6")
    centre = profile.index.map(lambda interval: (interval.left + interval.right) / 2)
    axes[0].plot(centre, profile["consommation"], color="#c00000", lw=2, label="moyenne par palier")
    axes[0].set_xlabel(f"prévision de température du jour cible ({TEMP_FORECAST}, °C)")
    axes[0].set_ylabel(f"{TARGET} (MW)")
    axes[0].set_title("Thermo-sensibilité : une courbe, pas une droite")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    heating = panel[panel[TEMP_FORECAST] < 14.0]
    cooling = panel[panel[TEMP_FORECAST] > 22.0]
    if len(heating) > 20 and len(cooling) > 20:
        slope_heat = np.polyfit(heating[TEMP_FORECAST], heating[TARGET], 1)[0]
        slope_cool = np.polyfit(cooling[TEMP_FORECAST], cooling[TARGET], 1)[0]
        axes[1].bar(
            ["chauffe (< 14 °C)", "climatisation (> 22 °C)"],
            [-slope_heat, slope_cool],
            color=["#c00000", "#2e75b6"],
        )
        axes[1].set_ylabel("MW par °C")
        axes[1].set_title("Gradient par régime (positif = la consommation monte)")
        axes[1].grid(alpha=0.3, axis="y")
        print(
            f"gradient de chauffe        : {slope_heat:7.1f} MW/°C "
            f"({100 * slope_heat / daily[TARGET].mean():+.2f} %/°C) sur {len(heating)} jours"
        )
        print(
            f"gradient de climatisation  : {slope_cool:7.1f} MW/°C "
            f"({100 * slope_cool / daily[TARGET].mean():+.2f} %/°C) sur {len(cooling)} jours"
        )
        print(f"rapport chauffe / clim     : {abs(slope_heat / slope_cool):.1f}x")
    fig.tight_layout()
    plt.show()

**Ce qu'il faut retenir**

- La relation est en **deux pentes** : forte et négative sous ~14 °C (chauffage), faible et positive au-dessus de ~22 °C (climatisation), plate entre les deux. Un modèle linéaire global la rate ; c'est la première raison d'utiliser des arbres.
- Le gradient de chauffe est plusieurs fois le gradient de climatisation : le parc est peu climatisé. Conséquence opérationnelle — une vague de froid coûte plus cher en erreur de prévision qu'une canicule de même amplitude.
- La dispersion autour de la moyenne par palier reste importante : elle contient l'effet du jour de la semaine et le bruit irréductible de la série. C'est cette dispersion qui fixe le plancher de MAPE atteignable.

### 5.3 Régimes exceptionnels : rares en jours, dominants en erreur

In [ ]:
# Régimes exceptionnels : ce qu'ils coûtent, et surtout ce qu'ils coûtent EN ERREUR.
if EVENT_COLUMN is None:
    print("aucune colonne de régime dans ce jeu : section sans objet")
else:
    panel = raw.drop_duplicates(subset=[TARGET_DATE]).copy()
    if NAIVE_COLUMN:
        panel["erreur_naive_pct"] = 100.0 * np.abs(
            (panel[TARGET] - panel[NAIVE_COLUMN]) / panel[TARGET]
        )
    regimes = panel.groupby(EVENT_COLUMN).agg(
        jours=(TARGET, "size"),
        consommation_moyenne=(TARGET, "mean"),
        consommation_max=(TARGET, "max"),
    )
    if NAIVE_COLUMN:
        regimes["erreur_naive_pct"] = panel.groupby(EVENT_COLUMN)["erreur_naive_pct"].mean()
    regimes["part_des_jours_pct"] = 100.0 * regimes["jours"] / regimes["jours"].sum()
    if NAIVE_COLUMN:
        total_error = (
            (panel["erreur_naive_pct"] / 100.0 * panel[TARGET]).groupby(panel[EVENT_COLUMN]).sum()
        )
        regimes["part_de_l_erreur_pct"] = 100.0 * total_error / total_error.sum()
    print(regimes.round(2).to_string())
    print()

    episodes = (
        panel.assign(jour=pd.to_datetime(panel[TARGET_DATE]).dt.date)
        .groupby([EVENT_COLUMN, "jour"], as_index=False)
        .size()
        .rename(columns={"size": "horizons"})
    )
    if len(episodes):
        grouped = (
            episodes[episodes[EVENT_COLUMN] != "none"]
            .groupby(EVENT_COLUMN)["jour"]
            .agg(jours="count", premier="min", dernier="max")
        )
        print("jours par régime (chaque date compte une fois, tous horizons confondus) :")
        print(grouped.to_string())

**Ce qu'il faut retenir**

- Les jours de régime exceptionnel représentent une faible part des lignes mais une part **disproportionnée de l'erreur totale** : ce sont eux qui justifient un indicateur de confiance et une revue humaine, pas le MAPE moyen.
- Le naif saisonnier est particulièrement mauvais pendant ces épisodes, parce qu'il recopie une semaine qui ne ressemblait pas à celle-ci. C'est là que le modèle appris gagne le plus — et là aussi qu'il peut perdre le plus s'il extrapole.
- Un arrêt industriel **baisse** la consommation : le traiter comme une anomalie à retrancher plutôt qu'à prévoir est une erreur de conception fréquente.

### 5.4 Audit du contrat d'antériorité

Cette section n'est pas décorative : c'est le test qui décide si le projet est honnête. Une
variable calculée avec la vérité du jour cible donne un MAPE proche de zéro en backtest et un
modèle **inutilisable** en production, puisque cette vérité n'existe pas encore à 6 h du matin.

In [ ]:
# AUDIT DU CONTRAT D'ANTÉRIORITÉ — la cellule la plus importante de ce notebook.
# Une feature est légale si et seulement si elle est connue à l'origine (date de publication) ou
# connue par avance (calendrier, prévision météo). Toute colonne qui dépend de la vérité du jour
# cible est une fuite : elle rend le MAPE excellent en backtest et le modèle inutilisable en réel.
panel = raw.copy()
panel["origine"] = pd.to_datetime(panel[TIME_COLUMN])
panel["cible"] = pd.to_datetime(panel[TARGET_DATE])
panel["ecart_jours"] = (panel["cible"] - panel["origine"]).dt.days

print("=== 1. cohérence de l'horizon déclaré ===")
print(panel["ecart_jours"].value_counts().sort_index().to_string())
if HORIZON_COLUMN in panel.columns:
    incoherent = int((panel["ecart_jours"] != panel[HORIZON_COLUMN]).sum())
    print(f"lignes où l'écart de dates diffère de `{HORIZON_COLUMN}` : {incoherent}")

print()
print("=== 2. classement des colonnes par disponibilité ===")
known_at_origin, known_in_advance, metadata, suspect = [], [], [], []
for column in panel.columns:
    if column in {"sample_id", TIME_COLUMN, TARGET_DATE, "ecart_jours", "origine", "cible"}:
        metadata.append(column)
    elif column.startswith("target_") or column in {"hdd_target", "cdd_target"}:
        known_in_advance.append(column)
    elif column == TARGET:
        metadata.append(column)
    elif column.startswith(("load_", "temperature_")):
        known_at_origin.append(column)
    else:
        suspect.append(column)
print(f"connues à l'origine (historique)     : {known_at_origin}")
print(f"connues par avance (calendrier/météo): {known_in_advance}")
print(f"métadonnées (jamais features)        : {metadata}")
print(f"à inspecter manuellement             : {suspect}")

print()
print("=== 3. vérification quantitative : les colonnes « historiques » datent-elles d'avant ? ===")
# `load_seasonal_naive` doit être la consommation du même jour de la semaine, UNE SEMAINE avant la
# cible. Si elle coïncide avec la cible, c'est une fuite ; on le mesure plutôt que de le supposer.
if NAIVE_COLUMN:
    daily = daily_series(raw).set_index(TARGET_DATE)[TARGET]
    expected = panel["cible"].map(lambda stamp: daily.get(stamp - pd.Timedelta(days=7), np.nan))
    match_naive = float(np.nanmean(np.isclose(panel[NAIVE_COLUMN], expected, rtol=1e-3)))
    match_target = float(np.nanmean(np.isclose(panel[NAIVE_COLUMN], panel[TARGET], rtol=1e-3)))
    print(f"`{NAIVE_COLUMN}` == valeur 7 jours avant la cible : {match_naive:.1%} des lignes")
    print(
        f"`{NAIVE_COLUMN}` == cible du jour             : {match_target:.1%} des lignes "
        "(doit rester bas : un taux élevé signerait une fuite)"
    )

if PERSIST_COLUMN:
    expected_last = panel["origine"].map(lambda stamp: daily.get(stamp, np.nan))
    match_persist = float(np.nanmean(np.isclose(panel[PERSIST_COLUMN], expected_last, rtol=1e-3)))
    print(f"`{PERSIST_COLUMN}` == valeur au jour d'origine    : {match_persist:.1%} des lignes")

print()
print("=== 4. la prévision météo est-elle entachée de son erreur ? ===")
# Une prévision météo parfaite serait une fuite déguisée : le modèle apprendrait une relation
# déterministe qui n'existe pas en exploitation. On vérifie qu'elle diffère de la réalisation.
if TEMP_FORECAST:
    anomalies = (panel[TEMP_FORECAST] - panel["temperature_anomaly_c"]).abs()
    print(f"écart prévision / normale saisonnière : moyenne={anomalies.mean():.2f} °C")
    print("dispersion de la prévision par horizon (°C, écart-type) :")
    print(panel.groupby(HORIZON_COLUMN)[TEMP_FORECAST].std().round(2).to_string())

**Ce qu'il faut retenir**

- Trois familles de colonnes, et une seule règle : **connue à l'origine** (historique de consommation, température observée), **connue par avance** (calendrier du jour cible, prévision météo) ou **métadonnée** (la cible elle-même, la date cible, le type de régime) — cette dernière est exclue de la matrice de features par `drop_columns`.
- La vérification est **quantitative** : le naif saisonnier coïncide avec la valeur de la semaine précédente, pas avec la cible du jour. Un taux de coïncidence élevé avec la cible signerait une fuite ; on le mesure au lieu de le supposer.
- Les colonnes `target_*` sont du calendrier : légitimes, parce que le calendrier d'un jour futur est connu. Les degrés-jours `hdd_target`/`cdd_target` sont calculés sur la **prévision** de température, jamais sur la réalisation — c'est le détail qui fait la différence entre une feature et une fuite.
- La prévision météo diffère de la réalisation, et son erreur croît avec l'horizon. Une prévision météo parfaite serait elle aussi une fuite déguisée.

### 5.5 Les références naïves : le plancher à battre

In [ ]:
# Les références naïves : le seul adversaire qui compte.
# Un modèle de prévision ne se juge pas à son MAPE absolu mais à son gain sur ce qu'un opérateur
# produit déjà sans modèle. Trois références, toutes construites avec l'information disponible à
# l'origine uniquement.
references = {}
if NAIVE_COLUMN:
    references["naif saisonnier (même jour, -7 j)"] = raw[NAIVE_COLUMN]
if PERSIST_COLUMN:
    references["persistance (dernière valeur connue)"] = raw[PERSIST_COLUMN]
for name, column in [
    ("moyenne glissante 7 j", "load_rolling_mean_7d"),
    ("climatologie 28 j", "load_rolling_mean_28d"),
]:
    if column in raw.columns:
        references[name] = raw[column]
references["moyenne globale (niveau constant)"] = pd.Series(
    float(raw[TARGET].mean()), index=raw.index
)

truth = raw[TARGET]
rows = []
for name, forecast in references.items():
    rows.append(
        {
            "référence": name,
            "MAPE %": round(mape(truth, forecast), 3),
            "MASE": round(
                mase(
                    truth, forecast, references.get("naif saisonnier (même jour, -7 j)", forecast)
                ),
                3,
            ),
            "MAE MW": round(float(np.mean(np.abs(truth - forecast))), 1),
            "biais %": round(float(np.mean((forecast - truth) / truth)) * 100.0, 2),
        }
    )
baseline_table = pd.DataFrame(rows).sort_values("MAPE %").reset_index(drop=True)
print(baseline_table.to_string(index=False))
print()
BEST_NAIVE = float(baseline_table["MAPE %"].iloc[0])
print(f"plancher à battre : {BEST_NAIVE:.2f} % de MAPE")

# Les critères de succès ne sont pas recopiés ici : ils sont lus dans le module de rapport, qui les
# tient lui-même du manifeste. Un seuil dupliqué dans un notebook finit toujours par diverger.
from src.evaluation.reports import DEFAULT_THRESHOLDS  # noqa: E402

print(
    f"critères déclarés : MAPE <= {DEFAULT_THRESHOLDS['mape_max']:.1f} %, "
    f"gain sur la meilleure référence >= "
    f"{DEFAULT_THRESHOLDS['improvement_vs_naive_min']:.0f} %, "
    f"MASE <= {DEFAULT_THRESHOLDS['mase_max']:.2f}"
)
print(
    f"à {BEST_NAIVE:.2f} % de MAPE naïf, le seuil de {DEFAULT_THRESHOLDS['mape_max']:.1f} % "
    f"exige un gain d'au moins "
    f"{100.0 * (1.0 - DEFAULT_THRESHOLDS['mape_max'] / BEST_NAIVE):.0f} %"
)

**Ce qu'il faut retenir**

- Le **naif saisonnier** (recopier la consommation du même jour de la semaine précédente) est la référence métier : c'est ce que fait l'opérateur sans modèle. Toute la valeur du projet se mesure en gain sur cette référence.
- La persistance est mauvaise dès que l'horizon s'allonge ou que le jour de la semaine change ; la climatologie 28 jours est mauvaise dès que la météo bouge. Chaque référence échoue sur une dimension différente — le modèle doit les battre toutes.
- Un gain inférieur à ~30 % ne justifie pas la complexité d'exploitation (astreinte, ré-entraînement, surveillance de dérive). C'est un critère de succès déclaré dans le manifeste, pas une opinion.

### 5.6 Le panel (origine, horizon)

In [ ]:
# La structure du panel : ce n'est pas une série, c'est un ensemble de problèmes de prévision.
counts = raw.groupby(HORIZON_COLUMN).size()
print(f"origines distinctes        : {raw[TIME_COLUMN].nunique()}")
print(f"jours cibles distincts     : {raw[TARGET_DATE].nunique()}")
print(f"lignes                     : {len(raw)}")
print(f"horizons publiés           : {sorted(counts.index.tolist())}")
print(counts.rename("lignes").to_string())
print()
overlap = raw.groupby(TARGET_DATE)[TIME_COLUMN].nunique()
print(
    f"jours cibles vus depuis plusieurs origines : {int((overlap > 1).sum())} (sur {len(overlap)})"
)
print()
# La même consommation est donc connue plusieurs fois, avec une information d'origine différente :
# c'est ce qui permet d'apprendre l'effet de l'horizon sans dupliquer l'information cible.
sample_day = raw[raw[TARGET_DATE] == raw[TARGET_DATE].max()]
columns = [TIME_COLUMN, TARGET_DATE, HORIZON_COLUMN, TARGET]
if PERSIST_COLUMN:
    columns.append(PERSIST_COLUMN)
print("le dernier jour cible, vu depuis chaque origine :")
print(sample_day[columns].sort_values(HORIZON_COLUMN).to_string(index=False))

**Ce qu'il faut retenir**

- Un même jour cible apparaît plusieurs fois, vu depuis des origines différentes : le modèle apprend donc **l'effet de l'horizon** comme une variable, au lieu d'entraîner un modèle par échéance.
- Ce choix a un coût (un compromis entre horizons) et un bénéfice (une seule chaîne à maintenir, plus de données par modèle). La section 3 du notebook 04 mesure les deux.
- Le split **chronologique sur l'origine** est obligatoire : couper aléatoirement placerait dans l'entraînement des lignes dont le jour cible est déjà dans le test, ce qui est une fuite par recouvrement de fenêtre.

## 6. Colinéarité et structure

In [ ]:
correlation = raw[numeric_columns].corr(numeric_only=True)
fig, axis = plt.subplots(figsize=(6.6, 5.4))
image = axis.imshow(correlation.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
axis.set_xticks(
    range(len(correlation.columns)), correlation.columns, rotation=45, ha="right", fontsize=7
)
axis.set_yticks(range(len(correlation.index)), correlation.index, fontsize=7)
for row in range(correlation.shape[0]):
    for column in range(correlation.shape[1]):
        value = correlation.iloc[row, column]
        axis.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=6,
            color="black" if abs(value) < 0.6 else "white",
        )
fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
axis.set_title("Corrélations de Pearson (variables numériques)")
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Deux features corrélées à > 0.9 n'apportent presque rien ensemble : en garder une simplifie le modèle et son explication.
- Les arbres sont robustes à la colinéarité ; les modèles linéaires/régularisés voient leurs coefficients devenir instables.
- La corrélation ne capture pas les relations **non linéaires** : la vérifier par des graphes cible vs feature.

In [ ]:
# Outliers : comptage par la règle de l'IQR (1.5 x écart interquartile).
rows = []
for column in numeric_columns:
    series = raw[column].dropna()
    if series.empty:
        continue
    low, high = series.quantile([0.25, 0.75])
    iqr = high - low
    outliers = int(((series < low - 1.5 * iqr) | (series > high + 1.5 * iqr)).sum())
    rows.append(
        {"colonne": column, "outliers_iqr": outliers, "part": outliers / max(len(series), 1)}
    )
outlier_frame = pd.DataFrame(rows).sort_values("outliers_iqr", ascending=False)
outlier_frame.head(8).round(4)

**Ce qu'il faut retenir**

- La règle IQR **signale**, elle ne tranche pas : un outlier peut être un client légitime (grand compte, pic saisonnier).
- Le winsorising (clip aux quantiles 1-99 %) conserve les lignes et les labels, contrairement à la suppression.

In [ ]:
# Intégrité : unicité de la clé et doublons complets.
key = CONFIG.data.id_column
duplicates = int(raw.duplicated().sum())
key_duplicates = int(raw[key].duplicated().sum()) if key and key in raw.columns else 0
print(f"doublons complets            : {duplicates}")
print(f"doublons sur la clé '{key}' : {key_duplicates}")
unique_keys = raw[key].nunique() if key in raw.columns else "n/a"
print(f"identifiants uniques         : {unique_keys} / {len(raw)}")

## 7. Synthèse de l'exploration

**Lectures clés de ce jeu de données**

- Le bâti réagit à une température lissée sur deux jours, pas à la seule valeur du jour cible : fournir la prévision de la veille (`temperature_forecast_prev_c`) et sa variation est ce qui permet au modèle de reconstruire cette inertie, et c'est un des rares gains disponibles au-delà du réglage des hyperparamètres.
- La consommation est pilotée à ~65 % par le calendrier et la température : un modèle qui ne connaît que les décalages temporels rate les épisodes de froid, un modèle qui ne connaît que la météo rate le creux du week-end. Les deux familles de features sont complémentaires, et le notebook 04 mesure leur apport séparé.
- L'erreur croît avec l'horizon (MAPE J+1 ≈ moitié du MAPE J+7) pour deux raisons distinctes : l'erreur de prévision météo augmente, et l'information de court terme (J-1) devient moins pertinente. Publier un MAPE unique sans ventilation par horizon masque cette structure.
- Le bruit est autocorrélé (AR(1) de coefficient ~0,6) : les résidus ne sont pas indépendants, donc les intervalles de confiance gaussiens naïfs sont trop étroits et le backtest par origine glissante est obligatoire — la validation croisée aléatoire produirait un score optimiste et faux.
- `temperature_lag_1d` et `temperature_rolling_mean_7d` comportent des manquants volontaires (~3 %, panne de capteur, surreprésentés en hiver) : l'imputation doit être explicite et un indicateur de manquant est rentable.
- Les jours fériés et les ponts produisent des creux de 10 à 20 % que le seul jour de la semaine ne prédit pas : sans feature calendaire explicite, le modèle sur-prévoit systématiquement ces jours-là.
- Le générateur injecte trois régimes de rupture (vague de froid, canicule, arrêt industriel) qui représentent ~4 % des jours cibles mais une part disproportionnée de l'erreur totale : c'est le cœur de l'analyse d'erreur du notebook 06. Les deux premiers déplacent réellement la température — un front lissé sur trois jours — donc le modèle peut les anticiper ; l'arrêt industriel est un choc de demande sans aucune signature météo, volontairement imprévisible.

### Décisions de modélisation issues de l'EDA

| Observation | Décision |
| --- | --- |
| Valeurs manquantes localisées | Imputation apprise sur le train (notebook 03) |
| Échelles hétérogènes | Scaling numérique obligatoire |
| Outliers légitimes | Winsorising plutôt que suppression |
| Modalités rares | Regroupement `rare` avant encodage |
| Colinéarité | Surveiller l'importance des features (notebook 04) |

**Suite** : `02_validation.ipynb` transforme ces observations en **contrats exécutables** (Pandera).